# Curated Trace Generation on H100

Generate TIR traces with gpt-oss-120b for 4,018 curated problems.
- **16 samples per problem** with logprobs
- Saves each problem as individual JSON (resume-safe)
- Skips already-completed problems on restart
- Time limit to stop gracefully before Kaggle kills the kernel

In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings; warnings.simplefilter('ignore')
import os, sys, subprocess, json, re, math, time, queue, threading, contextlib
from pathlib import Path
from collections import Counter

In [ ]:
def set_env(archive, tmp):
    if not os.path.exists(tmp):
        os.makedirs(tmp, exist_ok=True)
        subprocess.run(['tar', '-xzf', archive, '-C', tmp], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', f'{tmp}/wheels',
                    'unsloth', 'trl', 'vllm', 'openai_harmony'], check=True)

set_env('/kaggle/input/aimo-3-utils/wheels.tar.gz', '/kaggle/tmp/setup')

In [ ]:
for k, v in [('TRANSFORMERS_NO_TF', '1'), ('TRANSFORMERS_NO_FLAX', '1'), ('CUDA_VISIBLE_DEVICES', '0'),
             ('TOKENIZERS_PARALLELISM', 'false'), ('TRITON_PTXAS_PATH', '/usr/local/cuda/bin/ptxas'),
             ('TIKTOKEN_ENCODINGS_BASE', '/kaggle/tmp/setup/tiktoken_encodings')]:
    os.environ[k] = v

In [ ]:
from jupyter_client import KernelManager
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from openai import OpenAI
from openai_harmony import (HarmonyEncodingName, load_harmony_encoding, SystemContent, ReasoningEffort,
                             ToolNamespaceConfig, Author, Message, Role, TextContent, Conversation)
from transformers import set_seed

## Configuration

In [ ]:
class CFG:
    # Model
    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    kv_cache_dtype, dtype = 'fp8_e4m3', 'auto'

    # Trace generation
    n_samples = 16           # 16 samples per problem
    max_turns = 12
    temperature = 0.7
    top_logprobs = 10        # top-10 for better entropy estimates (research: 5 min, 20 ideal)

    # Prompts
    system_prompt = ('You are a world-class International Mathematical Olympiad (IMO) competitor. '
                    'The final answer must be a non-negative integer between 0 and 99999. '
                    'You must place the final integer answer inside \\boxed{}.')
    tool_prompt = ('Use this tool to execute Python code. The environment is a stateful Jupyter notebook. '
                  'You must use print() to output results.')
    preference_prompt = 'You have access to `math`, `numpy` and `sympy` to solve the problem.'

    # Timing
    server_timeout = 180
    sample_timeout = 300     # 5 min per sample max
    jupyter_timeout = 6
    time_limit = 8.5 * 3600  # Stop after 8.5 hours (Kaggle limit is 9h)

    # vLLM
    context_tokens = 65536
    buffer_tokens = 512
    gpu_memory_utilization = 0.96
    batch_size = 256
    min_p = 0.02
    seed = 42

    # Dataset
    problems_csv = '/kaggle/input/aimo3-curated-problems/problems.csv'

    # Output
    output_dir = '/kaggle/working/traces'

set_seed(CFG.seed)
os.makedirs(CFG.output_dir, exist_ok=True)
print(f'Generating {CFG.n_samples} samples per problem with {CFG.max_turns} max turns')
print(f'Time limit: {CFG.time_limit/3600:.1f} hours')
print(f'Top logprobs: {CFG.top_logprobs}')

## Load Problems from Dataset

In [ ]:
# Load curated problems from uploaded dataset
df = pd.read_csv(CFG.problems_csv)
PROBLEMS = []
for _, row in df.iterrows():
    answer = row.get('answer', '')
    # Try to parse integer answer
    try:
        answer = int(float(answer))
    except (ValueError, TypeError):
        answer = None
    PROBLEMS.append({
        'id': str(row['id']),
        'problem': str(row['problem']),
        'answer': answer,
        'topic': str(row.get('topic', '')),
        'source': str(row.get('source', '')),
    })

# Check for already-completed problems (resume support)
completed = set()
for f in Path(CFG.output_dir).glob('problem_*.json'):
    try:
        with open(f) as fh:
            data = json.load(fh)
            if len(data.get('samples', [])) == CFG.n_samples:
                completed.add(data['problem_id'])
    except:
        pass

remaining = [p for p in PROBLEMS if p['id'] not in completed]
print(f'Total problems: {len(PROBLEMS)}')
print(f'Already completed: {len(completed)}')
print(f'Remaining: {len(remaining)}')
print(f'Sources: {Counter(p["source"] for p in remaining).most_common()}')

## Sandbox and Tool Classes

In [ ]:
class AIMO3Template:
    def get_system_content(self, prompt, tool_cfg):
        return SystemContent.new().with_model_identity(prompt).with_reasoning_effort(
            reasoning_effort=ReasoningEffort.HIGH).with_tools(tool_cfg)

    def apply_chat_template(self, sys_prompt, usr_prompt, tool_cfg):
        return [Message.from_role_and_content(Role.SYSTEM, self.get_system_content(sys_prompt, tool_cfg)),
                Message.from_role_and_content(Role.USER, usr_prompt)]

In [ ]:
class AIMO3Sandbox:
    _port_lock, _next_port = threading.Lock(), 50000

    @classmethod
    def _get_next_ports(cls, count=5):
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout):
        self._default_timeout, self._owns_kernel, self._client, self._km = timeout, False, None, None
        ports = self._get_next_ports(5)
        env = os.environ.copy()
        env.update({'PYDEVD_DISABLE_FILE_VALIDATION': '1', 'PYDEVD_WARN_EVALUATION_TIMEOUT': '0',
                   'JUPYTER_PLATFORM_DIRS': '1', 'PYTHONWARNINGS': 'ignore', 'MPLBACKEND': 'Agg'})
        self._km = KernelManager()
        self._km.shell_port, self._km.iopub_port, self._km.stdin_port, self._km.hb_port, self._km.control_port = ports
        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])
        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True
        self.execute('import math, numpy, sympy, mpmath, itertools, collections\nmpmath.mp.dps = 64\n')

    def _format_error(self, tb):
        return ''.join(re.sub(r'\x1b\[[0-9;]*m', '', f) for f in tb
                      if 'File "' not in f or 'ipython-input' in f)

    def execute(self, code, timeout=None):
        effective_timeout = timeout or self._default_timeout
        msg_id = self._client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)
        stdout, stderr, start = [], [], time.time()
        while True:
            if time.time() - start > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'
            try:
                msg = self._client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue
            if msg.get('parent_header', {}).get('msg_id') != msg_id: continue
            mt, c = msg.get('msg_type'), msg.get('content', {})
            if mt == 'stream':
                (stdout if c.get('name') == 'stdout' else stderr).append(c.get('text', ''))
            elif mt == 'error':
                stderr.append(self._format_error(c.get('traceback', [])))
            elif mt in {'execute_result', 'display_data'}:
                if txt := c.get('data', {}).get('text/plain'):
                    stdout.append(txt if txt.endswith('\n') else f'{txt}\n')
            elif mt == 'status' and c.get('execution_state') == 'idle':
                break
        out, err = ''.join(stdout), ''.join(stderr)
        return f'{out.rstrip()}\n{err}' if err and out else (err or out or '[WARN] No output.')

    def close(self):
        with contextlib.suppress(Exception):
            if self._client: self._client.stop_channels()
        if self._owns_kernel and self._km:
            with contextlib.suppress(Exception): self._km.shutdown_kernel(now=True)
            with contextlib.suppress(Exception): self._km.cleanup_resources()

    def reset(self):
        self.execute('%reset -f\nimport math, numpy, sympy, mpmath, itertools, collections\nmpmath.mp.dps = 64\n')

In [ ]:
class AIMO3Tool:
    def __init__(self, timeout, prompt, sandbox):
        self._jupyter_timeout, self._tool_prompt, self._sandbox = timeout, prompt, sandbox
        self._lock = threading.Lock()

    def _ensure_last_print(self, code):
        lines = code.strip().split('\n')
        if not lines: return code
        last = lines[-1].strip()
        if any(x in last for x in ['print', 'import']) or not last or last.startswith('#'): return code
        lines[-1] = 'print(' + last + ')'
        return '\n'.join(lines)

    @property
    def tool_config(self): return ToolNamespaceConfig(name='python', description=self._tool_prompt, tools=[])

    def execute(self, code):
        with self._lock:
            return self._sandbox.execute(self._ensure_last_print(code))

## Trace Generator

In [ ]:
class TraceGenerator:
    def __init__(self, cfg, port=8000):
        self.cfg = cfg
        self.port = port
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()

        self._preload_weights()
        self._start_server()
        self.client = OpenAI(base_url=f'http://0.0.0.0:{port}/v1', api_key='sk-local', timeout=600)
        self._wait_for_server()
        self._init_sandbox()

    def _preload_weights(self):
        print(f'Preloading model weights from {self.cfg.model_path}...')
        start = time.time()
        files = []
        for root, _, fnames in os.walk(self.cfg.model_path):
            for fn in fnames:
                fp = os.path.join(root, fn)
                if os.path.isfile(fp): files.append(fp)
        with ThreadPoolExecutor(max_workers=8) as ex:
            list(ex.map(lambda p: open(p, 'rb').read(), files))
        print(f'Preloaded {len(files)} files in {time.time()-start:.1f}s')

    def _start_server(self):
        cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
               '--seed', str(self.cfg.seed), '--model', self.cfg.model_path,
               '--served-model-name', self.cfg.served_model_name,
               '--tensor-parallel-size', '1', '--max-num-seqs', str(self.cfg.batch_size),
               '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization),
               '--host', '0.0.0.0', '--port', str(self.port),
               '--dtype', self.cfg.dtype, '--kv-cache-dtype', self.cfg.kv_cache_dtype,
               '--max-model-len', str(self.cfg.context_tokens),
               '--async-scheduling', '--disable-log-stats', '--enable-prefix-caching']
        self.log_file = open('vllm_server.log', 'w')
        self.server = subprocess.Popen(cmd, stdout=self.log_file, stderr=subprocess.STDOUT, start_new_session=True)

    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start = time.time()
        for _ in range(self.cfg.server_timeout):
            if self.server.poll() is not None:
                raise RuntimeError(f'Server died: {open("vllm_server.log").read()}')
            try:
                self.client.models.list()
                print(f'Server ready in {time.time()-start:.1f}s')
                return
            except: time.sleep(1)
        raise RuntimeError('Server timeout')

    def _init_sandbox(self):
        print('Initializing sandbox...')
        self.sandbox = AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
        self.tool = AIMO3Tool(self.cfg.jupyter_timeout, self.cfg.tool_prompt, self.sandbox)
        print('Sandbox ready')

    def _scan_for_answer(self, text):
        for pattern in [r'\\boxed\s*\{\s*([0-9,]+)\s*\}', r'final\s+answer\s+is\s*([0-9,]+)']:
            if matches := re.findall(pattern, text, re.IGNORECASE):
                try:
                    val = int(matches[-1].replace(',', ''))
                    if 0 <= val <= 99999: return val
                except: pass
        return None

    def _compute_entropy(self, logprobs):
        """Compute mean Shannon entropy from a list of top-K logprob dicts."""
        if not logprobs: return float('inf')
        total, count = 0.0, 0
        for lp in logprobs:
            if isinstance(lp, dict) and lp:
                ent = sum(-math.exp(v)*math.log2(max(math.exp(v), 1e-30))
                          for v in lp.values() if math.exp(v) > 0)
                total += ent
                count += 1
        return total/count if count else float('inf')

    def _compute_per_token_entropy(self, logprobs):
        """Compute entropy at each token position. Returns list of floats."""
        entropies = []
        for lp in logprobs:
            if isinstance(lp, dict) and lp:
                ent = sum(-math.exp(v)*math.log2(max(math.exp(v), 1e-30))
                          for v in lp.values() if math.exp(v) > 0)
                entropies.append(ent)
        return entropies

    def _compute_ngram_rep(self, text, n=4):
        """Compute word-level n-gram repetition ratio. 0=no repetition, 1=all repeated."""
        words = text.split()
        if len(words) < n: return 0.0
        ngrams = [tuple(words[i:i+n]) for i in range(len(words)-n+1)]
        return 1.0 - len(set(ngrams)) / len(ngrams) if ngrams else 0.0

    def generate_sample(self, problem_text, sample_idx):
        """Generate a single sample trace with comprehensive metadata.

        Stores per-token signals for: PRIME (implicit PRM), DPO/KTO (preference),
        iw-SFT (importance weighting), Self-Certainty, DeepConf, PRM step rewards.
        """
        start = time.time()
        self.sandbox.reset()

        user_input = f'{problem_text} {self.cfg.preference_prompt}'
        conv = Conversation.from_messages(self.template.apply_chat_template(
            self.cfg.system_prompt, user_input, self.tool.tool_config))

        trace = {'turns': [], 'logprobs': []}
        answer = None
        seed = int((self.cfg.seed + sample_idx) ** 2)

        # Accumulate across all turns
        cumulative_logprob = 0.0       # sum of chosen token logprobs
        total_completion_tokens = 0
        chosen_logprobs_list = []      # per-token chosen logprob (for PRIME, DPO, Self-Certainty)
        finish_reason = None

        for turn in range(self.cfg.max_turns):
            prompt_ids = self.encoding.render_conversation_for_completion(conv, Role.ASSISTANT)
            prompt_tokens = len(prompt_ids)
            max_toks = self.cfg.context_tokens - prompt_tokens
            if max_toks < self.cfg.buffer_tokens: break

            try:
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name,
                    temperature=self.cfg.temperature,
                    logprobs=self.cfg.top_logprobs,
                    max_tokens=max_toks,
                    prompt=prompt_ids,
                    seed=seed,
                    stream=True,
                    extra_body={'min_p': self.cfg.min_p, 'stop_token_ids': self.stop_token_ids, 'return_token_ids': True}
                )

                tok_buf, txt_chunks, turn_logprobs = [], [], []
                turn_cumlogprob = 0.0
                turn_chosen_lps = []
                for chunk in stream:
                    if new_toks := chunk.choices[0].token_ids:
                        tok_buf.extend(new_toks)
                        txt_chunks.append(chunk.choices[0].text)
                        if (clp := chunk.choices[0].logprobs) and clp.top_logprobs:
                            for lp in clp.top_logprobs:
                                lp_dict = dict(lp)
                                turn_logprobs.append(lp_dict)
                                # Chosen token ≈ highest logprob in top-K
                                # (exact for greedy; close for temp=0.7)
                                if lp_dict:
                                    chosen_lp = max(lp_dict.values())
                                    turn_cumlogprob += chosen_lp
                                    turn_chosen_lps.append(chosen_lp)
                    fr = chunk.choices[0].finish_reason
                    if fr: finish_reason = fr
                    if '}' in chunk.choices[0].text:
                        if ans := self._scan_for_answer(''.join(txt_chunks[-32:])):
                            answer = ans
                            break
                stream.close()
            except Exception as e:
                trace['error'] = str(e)
                break

            if not tok_buf: break

            turn_tokens = len(tok_buf)
            turn_entropy = self._compute_entropy(turn_logprobs)
            full_text = ''.join(txt_chunks)

            trace['turns'].append({
                'role': 'assistant', 'content': full_text,
                'tokens': turn_tokens, 'entropy': turn_entropy,
                'cumulative_logprob': turn_cumlogprob,
            })
            trace['logprobs'].extend(turn_logprobs)
            cumulative_logprob += turn_cumlogprob
            total_completion_tokens += turn_tokens
            chosen_logprobs_list.extend(turn_chosen_lps)

            if answer: break

            # Parse and handle tool calls
            new_msgs = self.encoding.parse_messages_from_completion_tokens(tok_buf, Role.ASSISTANT)
            conv.messages.extend(new_msgs)
            last = new_msgs[-1]

            if last.channel == 'final':
                answer = self._scan_for_answer(last.content[0].text)
                break

            if last.recipient == 'python':
                code = last.content[0].text
                output = self.tool.execute(code)
                code_success = '[ERROR]' not in output and '[WARN]' not in output
                trace['turns'].append({
                    'role': 'tool', 'name': 'python', 'content': output,
                    'code_success': code_success,
                })
                conv.messages.append(Message(
                    author=Author(role=Role.TOOL, name='python'),
                    content=[TextContent(text=output)]
                ).with_recipient('assistant'))

        # ============================================================
        # Per-sample metadata — comprehensive for all downstream uses
        # ============================================================

        # --- Core ---
        trace['answer'] = answer
        trace['finish_reason'] = finish_reason
        trace['time'] = time.time() - start
        trace['n_turns'] = len([t for t in trace['turns'] if t['role'] == 'assistant'])

        # --- Logprob signals (PRIME, DPO, KTO, iw-SFT, Self-Certainty) ---
        trace['cumulative_logprob'] = cumulative_logprob
        trace['completion_tokens'] = total_completion_tokens
        trace['length_normalized_logprob'] = (
            cumulative_logprob / total_completion_tokens if total_completion_tokens > 0 else float('-inf'))
        trace['chosen_logprobs'] = chosen_logprobs_list  # per-token, for PRIME pi_theta

        # --- Entropy signals (DeepConf, PRM step rewards, Think Just Enough) ---
        per_token_ents = self._compute_per_token_entropy(trace['logprobs'])
        trace['entropy'] = sum(per_token_ents) / len(per_token_ents) if per_token_ents else float('inf')
        trace['per_token_entropy'] = per_token_ents  # full trajectory for DeepConf group analysis
        if per_token_ents:
            trace['entropy_std'] = (sum((e - trace['entropy'])**2 for e in per_token_ents) / len(per_token_ents)) ** 0.5
            trace['entropy_min'] = min(per_token_ents)
            trace['entropy_max'] = max(per_token_ents)
            sorted_ents = sorted(per_token_ents)
            p10_idx = max(0, len(sorted_ents) // 10 - 1)
            trace['entropy_p10'] = sorted_ents[p10_idx]  # DeepConf bottom-10%
        else:
            trace['entropy_std'] = 0.0
            trace['entropy_min'] = float('inf')
            trace['entropy_max'] = 0.0
            trace['entropy_p10'] = float('inf')

        # --- Code execution signals (CodePRM, TIR preference, GRPO format reward) ---
        trace['code_calls'] = sum(1 for t in trace['turns'] if t.get('role') == 'tool')
        trace['code_errors'] = sum(
            1 for t in trace['turns'] if t.get('role') == 'tool' and not t.get('code_success', True))
        trace['code_success_rate'] = (
            (trace['code_calls'] - trace['code_errors']) / trace['code_calls']
            if trace['code_calls'] > 0 else 1.0)

        # --- Answer stability (TIR preference, quality filtering) ---
        all_turn_answers = []
        full_text_parts = []
        for t in trace['turns']:
            if t['role'] == 'assistant':
                full_text_parts.append(t['content'])
                ans = self._scan_for_answer(t['content'])
                if ans is not None:
                    all_turn_answers.append(ans)
        trace['all_turn_answers'] = all_turn_answers
        trace['answer_changed_count'] = sum(
            1 for i in range(1, len(all_turn_answers)) if all_turn_answers[i] != all_turn_answers[i-1])

        # --- Format validity (GRPO format reward, DPO-VP) ---
        full_text = ''.join(full_text_parts)
        trace['answer_format_valid'] = bool(re.search(r'\\boxed\s*\{', full_text))

        # --- Degeneration detection (repetition filtering) ---
        trace['ngram_rep_4'] = self._compute_ngram_rep(full_text, n=4)

        return trace

    def generate_traces(self, problem):
        """Generate all samples for a problem. Saves JSON immediately."""
        problem_id = problem['id']
        problem_text = problem['problem']
        ground_truth = problem.get('answer')
        safe_id = re.sub(r'[^a-zA-Z0-9_]', '_', problem_id)
        output_path = Path(self.cfg.output_dir) / f'problem_{safe_id}.json'

        print(f"\n{'='*60}")
        print(f'Problem: {problem_id}')
        print(f'Ground truth: {ground_truth}')
        print(f"{'='*60}")

        samples = []
        for i in range(self.cfg.n_samples):
            trace = self.generate_sample(problem_text, i)
            samples.append(trace)
            status = 'OK' if trace['answer'] == ground_truth else ('WRONG' if trace['answer'] else 'NONE')
            print(f"  Sample {i+1}/{self.cfg.n_samples}: answer={trace['answer']} "
                  f"(ent={trace['entropy']:.3f} clp={trace['cumulative_logprob']:.1f} "
                  f"toks={trace['completion_tokens']} turns={trace['n_turns']} "
                  f"code={trace['code_calls']}/{trace['code_errors']}err "
                  f"rep4={trace['ngram_rep_4']:.2f} "
                  f"t={trace['time']:.1f}s) [{status}]")

        # --- Per-sample correctness label (needed by ALL training methods) ---
        for s in samples:
            if ground_truth is not None and s['answer'] is not None:
                s['is_correct'] = (s['answer'] == ground_truth)
            else:
                s['is_correct'] = None

        # --- Problem-level aggregates (GRPO, iw-SFT difficulty weighting) ---
        answers = [s['answer'] for s in samples if s['answer'] is not None]
        votes = Counter(answers)
        n_correct = sum(1 for s in samples if s.get('is_correct') is True)

        output = {
            'problem_id': problem_id,
            'problem': problem_text,
            'ground_truth': ground_truth,
            'topic': problem.get('topic', ''),
            'source': problem.get('source', ''),
            'samples': samples,
            'model': self.cfg.served_model_name,
            'n_samples': self.cfg.n_samples,
            'temperature': self.cfg.temperature,
            'top_logprobs': self.cfg.top_logprobs,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            # Problem-level aggregates
            'n_correct': n_correct,
            'pass_rate': n_correct / len(samples) if samples else 0.0,
            'answer_distribution': dict(votes),
        }

        # Compact JSON (no indent) — traces can be large with per-token data
        with open(output_path, 'w') as f:
            json.dump(output, f)
        print(f'  Saved to {output_path}')

        # Summary
        if answers:
            top_answer = votes.most_common(1)[0][0]
            correct = top_answer == ground_truth if ground_truth is not None else 'N/A'
            print(f'  >> Majority={top_answer}, Votes={dict(votes)}, '
                  f'Correct={correct}, PassRate={n_correct}/{len(samples)}')

        return output

    def cleanup(self):
        self.sandbox.close()
        self.server.terminate()
        self.server.wait()
        self.log_file.close()

## Generate Traces

In [ ]:
generator = TraceGenerator(CFG)

In [ ]:
notebook_start = time.time()
completed_count = 0
stopped_reason = 'finished'

for i, problem in enumerate(remaining):
    # Check time limit before starting a new problem
    elapsed = time.time() - notebook_start
    if elapsed > CFG.time_limit:
        stopped_reason = f'time_limit ({elapsed/3600:.1f}h)'
        print(f'\n*** TIME LIMIT REACHED ({elapsed/3600:.1f}h) — stopping gracefully ***')
        break

    print(f'\n[{i+1}/{len(remaining)}] Processing {problem["id"]}...')
    try:
        result = generator.generate_traces(problem)
        completed_count += 1
    except Exception as e:
        print(f'  ERROR: {e}')
        # Save partial result so we don't lose everything
        safe_id = re.sub(r'[^a-zA-Z0-9_]', '_', problem['id'])
        error_path = Path(CFG.output_dir) / f'error_{safe_id}.json'
        with open(error_path, 'w') as f:
            json.dump({'problem_id': problem['id'], 'error': str(e)}, f)
        continue

    elapsed = time.time() - notebook_start
    avg_time = elapsed / (i + 1)
    eta = avg_time * (len(remaining) - i - 1)
    print(f'\nProgress: {i+1}/{len(remaining)} | Elapsed: {elapsed/60:.1f}min | '
          f'ETA: {eta/60:.1f}min ({eta/3600:.1f}h) | Completed this session: {completed_count}')

total_elapsed = time.time() - notebook_start
total_completed = len(list(Path(CFG.output_dir).glob('problem_*.json')))
print(f"\n{'='*60}")
print(f'Session complete: {completed_count} problems in {total_elapsed/60:.1f} minutes')
print(f'Stopped: {stopped_reason}')
print(f'Total completed (all sessions): {total_completed}/{len(PROBLEMS)}')
print(f'Output dir: {CFG.output_dir}')
print(f"{'='*60}")

In [ ]:
# Summary of all completed traces
import glob
trace_files = sorted(glob.glob(f'{CFG.output_dir}/problem_*.json'))
print(f'Total trace files: {len(trace_files)}')

correct, wrong, no_answer, total = 0, 0, 0, 0
for tf in trace_files:
    with open(tf) as f:
        data = json.load(f)
    gt = data.get('ground_truth')
    answers = [s['answer'] for s in data['samples'] if s['answer'] is not None]
    total += 1
    if not answers:
        no_answer += 1
        continue
    majority = Counter(answers).most_common(1)[0][0]
    if gt is not None:
        if majority == gt:
            correct += 1
        else:
            wrong += 1
    else:
        no_answer += 1  # no ground truth to compare

print(f'With ground truth: {correct} correct, {wrong} wrong')
print(f'No answer / no ground truth: {no_answer}')
if correct + wrong > 0:
    print(f'Accuracy (majority vote): {correct/(correct+wrong)*100:.1f}%')

In [ ]:
generator.cleanup()
print('Cleanup complete')